# 10 – Ratings

Esplorazione e data cleaning del dataset `ratings.csv`.

| Colonna | Descrizione |
|---|---|
| `username` | Nome utente MAL |
| `anime_id` | ID dell'anime su MAL |
| `status` | Stato di visione |
| `score` | Voto dell'utente |
| `is_rewatching` | Indica se l'utente sta rivedendo l'anime |
| `num_watched_episodes` | Numero di episodi guardati |

## 1. Import e caricamento dati

Importiamo le librerie necessarie e carichiamo il file csv. Facciamo una esplorazione generica per capire la struttura e le caratteristiche del dataset.

In [ ]:
import pandas as pd
import numpy as np
from dataset_analyzer import analyze
from foreign_key_analyzer import check_fk

df_rt = pd.read_csv('../datasets/ratings.csv')
print(f'Shape: {df_rt.shape}')
df_rt.info(show_counts=True)
df_rt.head()

**Osservazioni iniziali:**
- Il dataset contiene 124,298,357 di righe e 6 colonne.
- `username` presenta 7 righe nulle.
- `is_rewatching` contiene 3,797,321 valori null che controlliamo successivamente per verificare se si tratta di un errore o meno.

## 1.1 Rimozione duplicati esatti

Prima dell'analisi per colonna, rimuoviamo le righe con valori identici in **tutte** le colonne, mantenendo solo la prima occorrenza.

La dimensione del dataset (~124M righe) rende la verifica tramite `df.duplicated()` proibitiva in termini di memoria. Usiamo `pd.util.hash_pandas_object()` che calcola un hash `uint64` per ogni riga e cerca duplicati su quella serie, evitando di confrontare le righe intere.

In [ ]:
n_originale = len(df_rt)

hashes = pd.util.hash_pandas_object(df_rt, index=False)
n_dup = hashes.duplicated(keep=False).sum()
print(f'Duplicati esatti sull\'intero dataset: {n_dup:,}')
if n_dup == 0:
    print('→ Nessun duplicato esatto, nessuna operazione richiesta.')
else:
    print('→ Presenza di duplicati: rimozione necessaria.')
    df_rt = df_rt[~hashes.duplicated(keep='first')].reset_index(drop=True)
    print(f'Righe dopo rimozione: {len(df_rt):,}')

Nessun duplicato esatto trovato. Tutte le righe sono già uniche. Il dataset rimane invariato.

Adesso che siamo sicuri che tutte le righe sono uniche, iniziamo l'analisi per colonne utilizzando la nostra libreria `dataset_analyzer`.